# qdmpy Quickstart

**Target user**: "I want to fit and be done."

This notebook shows the minimal path from data to B111 field maps.

## With real data

```python
import qdmpy
result = qdmpy.load('/path/to/FOV').fit_odmr()
```

## This notebook uses synthetic data

Because real `.mat` files and GPU fitting are not required in CI,
we use `qdmpy.make_synthetic_qdm_result()` which creates a realistic
result object without loading files or running the GPU fitter.

In [ ]:
%matplotlib inline
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

import qdmpy

## 1. Load / create a result

In real usage replace `make_synthetic_qdm_result(...)` with:
```python
result = qdmpy.load('/data/FOV18x').fit_odmr()
```

In [ ]:
result = qdmpy.make_synthetic_qdm_result(shape=(32, 32), model_name='ESR14N')

## 2. B111 magnetic field maps

- **Remanent** field: permanent magnetisation (ferro component)
- **Induced** field: paramagnetic / bias-tracking component

In [ ]:
b111_rem = result.b111_remanent   # 2D numpy array, µT
b111_ind = result.b111_induced


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, data, title in zip(axes,
    [b111_rem, b111_ind],
    ['B111 remanent (µT)', 'B111 induced (µT)'], strict=False):
    vmax = np.percentile(np.abs(data), 98)
    im = ax.imshow(data, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 3. B111 as xarray Dataset

For labelled access and easy export use `result.b111` (xr.Dataset).

In [ ]:
b111_ds = result.b111

## 4. 3D field reconstruction (Bx, By, Bz)

The `magnetic_map` property applies Fourier-domain inversion to recover
the full 3D field vector from B111.

In [ ]:
mm = result.magnetic_map

## 5. Save and reload

`QDMResult.save()` / `QDMResult.load()` round-trips via NPZ.

In [ ]:
import pathlib
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = pathlib.Path(tmp) / 'result.npz'
    result.save(path)
    reloaded = qdmpy.QDMResult.load(path)


## Summary

| API | What you get |
|-----|--------------|
| `qdmpy.load(path).fit_odmr()` | `QDMResult` from real data |
| `result.b111_remanent` | 2D ndarray in µT |
| `result.b111_induced` | 2D ndarray in µT |
| `result.b111` | `xr.Dataset` with 'remanent' and 'induced' |
| `result.magnetic_map` | Full Bx/By/Bz/Btotal via Fourier inversion |
| `result.save(path)` | Save to NPZ |
| `QDMResult.load(path)` | Reload from NPZ |